In [8]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score



Load data


In [9]:
df = pd.read_csv("house_prices.csv")
print(df.shape)


(1460, 81)


Drop the 2 outliers found in EDA

In [10]:
df = df[~((df["GrLivArea"] > 4000) & (df["SalePrice"] < 300000))]
print(df.shape)   

(1458, 81)


Drop Id (just a row number, not useful for prediction)

In [11]:
df = df.drop("Id", axis=1)

In [12]:
print(df.shape)

(1458, 80)


NaN here means "house doesn't have this feature" -> fill with 'None'


In [13]:
none_cols = ["PoolQC", "Alley", "Fence", "FireplaceQu", "MiscFeature", "MasVnrType",
             "GarageType", "GarageFinish", "GarageQual", "GarageCond",
             "BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2"]
df[none_cols] = df[none_cols].fillna("None")


Any other missing text values (like Electrical) also become 'None'


In [14]:
cat_cols = df.select_dtypes(exclude="number").columns
df[cat_cols] = df[cat_cols].fillna("None")



Target: log of SalePrice (fixes the skew found in EDA)

In [15]:
y = np.log1p(df["SalePrice"])


Features: drop the target and one column from each correlated pair


In [16]:
X = df.drop(["SalePrice", "GarageArea", "1stFlrSF"], axis=1)


Convert text columns to 0/1 columns


In [17]:
X = pd.get_dummies(X, drop_first=True, dtype=int)
print("Features after get_dummies:", X.shape)


Features after get_dummies: (1458, 257)


Split into train and test


In [18]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)


Fill missing numeric values (LotFrontage, MasVnrArea, GarageYrBlt) with the TRAIN median


In [19]:
medians = X_train.median()
X_train = X_train.fillna(medians)
X_test = X_test.fillna(medians)


Scale (on train only)


In [21]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)


Train linear regression

In [22]:
lin_model = LinearRegression()
lin_model.fit(X_train_s, y_train)


,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


Predict (these are in log scale), then convert back to real prices


In [23]:
pred_log = lin_model.predict(X_test_s)
pred = np.expm1(pred_log)
actual = np.expm1(y_test)



Evaluate

In [25]:
print("R2 (log scale):", r2_score(y_test, pred_log))
print("R2 (price):", r2_score(actual, pred))
print("MAE ($):", mean_absolute_error(actual, pred))
print("RMSE ($):", np.sqrt(mean_squared_error(actual, pred)))

R2 (log scale): 0.890348393938167
R2 (price): 0.9137476229077418
MAE ($): 15586.338122823327
RMSE ($): 21827.38223133004


MAE : on average, a prediction is off by about $15.6k

1. "$15.6k error is about 8-9%"

MAE = $15,586 means your model's guess is wrong by about $15,586 on average. That number alone doesn't tell you whether it's good or bad, so we compare it with the size of a normal house price.

The average house in this dataset costs about $180,000. Compare the error with that:

15,586 / 180,000 ≈ 0.087, which is about 8.7%

So if a house really costs $180,000, your model usually guesses somewhere around $164,000 to $196,000. For a simple model, that's decent.

2. "RMSE is higher than MAE"

Both measure error, but they treat big mistakes differently.

MAE treats every mistake equally.
RMSE squares each error before averaging, so a big mistake counts for much more.

Example with 3 houses, where the model is off by $5k, $5k and $50k:

MAE = (5 + 5 + 50) / 3 = $20k
RMSE = √((25 + 25 + 2500) / 3) = √850 = $29k

The one big miss ($50k) pushed RMSE up much more than MAE. In your results, RMSE ($21.8k) is higher than MAE ($15.6k), which tells you that most predictions are quite close, but a few are off by a lot. Those few are usually expensive houses, where even a small percentage error means thousands of dollars.

Whenever RMSE is much larger than MAE, it means there are some big misses.

3. "R² on price is higher than on log scale"

R² answers: how much of the difference between houses does the model explain?

On the log scale, expensive and cheap houses are squeezed closer together, so all houses matter about equally.
On the price scale, expensive houses have huge price gaps ($500k vs $150k), so they matter much more.

MAE = the average mistake in dollars, and comparing it with the average price shows it's about 8-9%.


RMSE is bigger than MAE because it punishes big mistakes more, so a few predictions are far off.


R² is a score out of 1, and 0.91 means the model explains about 91% of the price differences.